<a href="https://colab.research.google.com/github/noorarora/ARTI6000-RLHF-Assignment/blob/main/assignment1_rlhf/notebooks/04_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

!pip install -q transformers datasets accelerate sentencepiece


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [4]:
sft_model_path = "/content/drive/MyDrive/rlhf_assignment/models/sft_baseline_distilgpt2"
dpo_model_path = "/content/drive/MyDrive/rlhf_assignment/models/dpo_aligned_distilgpt2"
reward_model_path = "/content/drive/MyDrive/rlhf_assignment/models/reward_model_distilbert"

In [5]:
sft_tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_path)

if sft_tokenizer.pad_token is None:
    sft_tokenizer.pad_token = sft_tokenizer.eos_token

In [6]:
sft_model = AutoModelForCausalLM.from_pretrained(sft_model_path).to(device)
dpo_model = AutoModelForCausalLM.from_pretrained(dpo_model_path).to(device)
reward_model = AutoModelForSequenceClassification.from_pretrained(reward_model_path).to(device)

sft_model.eval()
dpo_model.eval()
reward_model.eval()

print("All models loaded successfully.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

All models loaded successfully.


In [7]:
test_prompts = [
    "Human: Explain reinforcement learning in simple words.\n\nAssistant:",
    "Human: Give three tips for effective teamwork.\n\nAssistant:",
    "Human: How can a student manage time better during exams?\n\nAssistant:"
]

In [8]:
def generate_response(model, prompt):
    inputs = sft_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=sft_tokenizer.eos_token_id
        )

    return sft_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [9]:
def reward_score(text):
    inputs = reward_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        score = reward_model(**inputs).logits.item()

    return score

In [10]:
for prompt in test_prompts:
    sft_output = generate_response(sft_model, prompt)
    dpo_output = generate_response(dpo_model, prompt)

    sft_score = reward_score(sft_output)
    dpo_score = reward_score(dpo_output)

    print("\n" + "=" * 80)
    print("PROMPT:")
    print(prompt)

    print("\n--- SFT RESPONSE ---")
    print(sft_output)
    print("Reward score:", sft_score)

    print("\n--- DPO RESPONSE ---")
    print(dpo_output)
    print("Reward score:", dpo_score)


PROMPT:
Human: Explain reinforcement learning in simple words.

Assistant:

--- SFT RESPONSE ---
Human: Explain reinforcement learning in simple words.

Assistant: What can we learn from reinforcement learning?
Response: We can learn from learning from reinforcement learning techniques. For example, we can learn from reinforcement learning techniques such as reinforcement learning, and we can learn from a variety of strategies, including reinforcement learning, and reinforcement learning. We can learn from reinforcement learning techniques such as reinforcement learning, and we can learn from strategies such as reinforcement learning, and we can
Reward score: 0.1335860937833786

--- DPO RESPONSE ---
Human: Explain reinforcement learning in simple words.

Assistant: In addition to reinforcement learning, learning is used to make predictions about the behavior of a given animal. It can be done in a variety of ways, including to predict the future and predict food, weather, and other fac

In [11]:
import pandas as pd

results = []

for prompt in test_prompts:
    sft_output = generate_response(sft_model, prompt)
    dpo_output = generate_response(dpo_model, prompt)

    sft_score = reward_score(sft_output)
    dpo_score = reward_score(dpo_output)

    winner = "DPO" if dpo_score > sft_score else "SFT"

    results.append({
        "prompt": prompt,
        "sft_reward_score": sft_score,
        "dpo_reward_score": dpo_score,
        "winner": winner,
        "sft_output": sft_output,
        "dpo_output": dpo_output
    })

results_df = pd.DataFrame(results)
results_df[["prompt", "sft_reward_score", "dpo_reward_score", "winner"]]

,prompt,sft_reward_score,dpo_reward_score,winner
0,Human: Explain reinforcement learning in simpl...,0.114149,0.129138,DPO
1,Human: Give three tips for effective teamwork....,0.111654,0.056698,SFT
2,Human: How can a student manage time better du...,0.146706,0.093462,SFT


In [12]:
dpo_wins = (results_df["winner"] == "DPO").sum()
sft_wins = (results_df["winner"] == "SFT").sum()

print("DPO wins:", dpo_wins)
print("SFT wins:", sft_wins)

if dpo_wins > sft_wins:
    print("Overall, the DPO aligned model performed better according to the reward model.")
elif sft_wins > dpo_wins:
    print("Overall, the SFT baseline model performed better according to the reward model.")
else:
    print("Both models performed equally on this evaluation set.")

DPO wins: 1
SFT wins: 2
Overall, the SFT baseline model performed better according to the reward model.


In [14]:
import os

results_dir = "/content/drive/MyDrive/rlhf_assignment/results"
os.makedirs(results_dir, exist_ok=True)

In [15]:
results_path = "/content/drive/MyDrive/rlhf_assignment/results/evaluation_results.csv"

results_df.to_csv(results_path, index=False)

print("Results saved to:", results_path)

Results saved to: /content/drive/MyDrive/rlhf_assignment/results/evaluation_results.csv
